# 🤖 Build a Sample Chatbot using Amazon Lex
**K21Academy Hands-On Lab**

Run each cell in order to automatically create, configure, build, test, and clean up a chatbot on Amazon Lex.

> Works on **AWS SageMaker** (credentials auto-configured via execution role).

## Step 0 — Install & Import Dependencies

In [ ]:
!pip install boto3 --quiet

In [ ]:
import boto3
import botocore
import json
import time

# Initialize clients
lex_client = boto3.client('lexv2-models', region_name='us-east-1')
sts_client = boto3.client('sts')

print('✅ boto3 clients initialized')

## Step 1 — Get the SageMaker Execution Role

On SageMaker, we reuse the existing execution role instead of creating a new one.
This avoids IAM permission errors.

In [ ]:
# Derive the IAM role ARN from the current SageMaker session
identity = sts_client.get_caller_identity()
caller_arn = identity['Arn']  # e.g. arn:aws:sts::123456789:assumed-role/AmazonSageMaker-ExecutionRole-xxx/SageMaker

# Convert assumed-role ARN -> actual IAM role ARN
# From: arn:aws:sts::ACCOUNT:assumed-role/ROLE-NAME/session
# To:   arn:aws:iam::ACCOUNT:role/ROLE-NAME
parts = caller_arn.replace(':sts:', ':iam:').replace('assumed-role', 'role').split('/')
ROLE_ARN = '/'.join(parts[:2])  # drops the session name at the end

print(f'✅ Using IAM Role ARN: {ROLE_ARN}')

## Step 2 — Create the Bot

In [ ]:
BOT_NAME = 'mySampleChatBot'

response = lex_client.create_bot(
    botName=BOT_NAME,
    description='Chatbot for handling greetings and food orders.',
    roleArn=ROLE_ARN,
    dataPrivacy={'childDirected': False},
    idleSessionTTLInSeconds=300
)

BOT_ID = response['botId']
print(f'✅ Bot created!')
print(f'   Bot Name : {BOT_NAME}')
print(f'   Bot ID   : {BOT_ID}')

## Step 3 — Create a Bot Locale (English)

In [ ]:
LOCALE_ID = 'en_US'
BOT_VERSION = 'DRAFT'

lex_client.create_bot_locale(
    botId=BOT_ID,
    botVersion=BOT_VERSION,
    localeId=LOCALE_ID,
    nluIntentConfidenceThreshold=0.40,
    description='English locale for mySampleChatBot'
)

print(f'✅ Bot locale created: {LOCALE_ID} (confidence threshold: 0.40)')

## Step 4 — Create GreetingIntent

Handles: `Hello`, `Hi`, `Hai`  
Responds with: **"Hi, what would you like to order?"**

In [ ]:
response = lex_client.create_intent(
    intentName='GreetingIntent',
    description="Handles user greetings like 'Hello', 'Hi', etc.",
    sampleUtterances=[
        {'utterance': 'Hello'},
        {'utterance': 'Hi'},
        {'utterance': 'Hai'},
        {'utterance': 'Hey'},
        {'utterance': 'Good morning'},
    ],
    initialResponseSetting={
        'initialResponse': {
            'messageGroups': [
                {
                    'message': {
                        'plainTextMessage': {
                            'value': 'Hi, what would you like to order?'
                        }
                    }
                }
            ],
            'allowInterrupt': True
        }
    },
    botId=BOT_ID,
    botVersion=BOT_VERSION,
    localeId=LOCALE_ID
)

GREETING_INTENT_ID = response['intentId']
print(f'✅ GreetingIntent created! (ID: {GREETING_INTENT_ID})')
print(f'   Response: "Hi, what would you like to order?"')

## Step 5 — Create OrderFoodIntent

Handles: `I want to order 1 pizza`, `I want to order 2 Burgers`  
Responds with: **"Your order has been placed"**

In [ ]:
response = lex_client.create_intent(
    intentName='OrderFoodIntent',
    description="Handles food orders, e.g., 'I want to order pizza.'",
    sampleUtterances=[
        {'utterance': 'I want to order 1 pizza'},
        {'utterance': 'I want to order 2 Burgers'},
        {'utterance': 'Can I order food'},
        {'utterance': 'I would like to order'},
        {'utterance': 'Please take my order'},
    ],
    initialResponseSetting={
        'initialResponse': {
            'messageGroups': [
                {
                    'message': {
                        'plainTextMessage': {
                            'value': 'Your order has been placed'
                        }
                    }
                }
            ],
            'allowInterrupt': True
        }
    },
    botId=BOT_ID,
    botVersion=BOT_VERSION,
    localeId=LOCALE_ID
)

ORDER_INTENT_ID = response['intentId']
print(f'✅ OrderFoodIntent created! (ID: {ORDER_INTENT_ID})')
print(f'   Response: "Your order has been placed"')

## Step 5b — Configure FallbackIntent

Responds with: **"Sorry, I can't help with that."** for any unrecognised input.

In [ ]:
# Find the auto-created FallbackIntent
intents = lex_client.list_intents(
    botId=BOT_ID,
    botVersion=BOT_VERSION,
    localeId=LOCALE_ID
)['intentSummaries']

fallback = next(i for i in intents if i['intentName'] == 'FallbackIntent')
FALLBACK_INTENT_ID = fallback['intentId']

# Update it with a response message
# Note: AMAZON.FallbackIntent only supports 'intentClosingSetting' (not
# 'initialResponseSetting'), and 'intentName' must be supplied as-is.
lex_client.update_intent(
    intentId=FALLBACK_INTENT_ID,
    intentName='FallbackIntent',
    botId=BOT_ID,
    botVersion=BOT_VERSION,
    localeId=LOCALE_ID,
    intentClosingSetting={
        'closingResponse': {
            'messageGroups': [
                {
                    'message': {
                        'plainTextMessage': {
                            'value': "Sorry, I can't help with that."
                        }
                    }
                }
            ],
            'allowInterrupt': True
        }
    }
)

print(f'✅ FallbackIntent configured! (ID: {FALLBACK_INTENT_ID})')
print(f'   Response: "Sorry, I can\'t help with that."')


## Step 6 — Build the Bot

This cell triggers the build and waits until it completes (polls every 15 seconds).

In [ ]:
lex_client.build_bot_locale(
    botId=BOT_ID,
    botVersion=BOT_VERSION,
    localeId=LOCALE_ID
)

print('⏳ Building bot... polling every 15 seconds.')

while True:
    status = lex_client.describe_bot_locale(
        botId=BOT_ID,
        botVersion=BOT_VERSION,
        localeId=LOCALE_ID
    )['botLocaleStatus']

    print(f'   Status: {status}')

    if status == 'Built':
        print('✅ Bot built successfully!')
        break
    elif status in ['Failed', 'NotBuilt']:
        print(f'❌ Build failed with status: {status}')
        break

    time.sleep(15)

## Step 7 — Create a Numbered Bot Version & Alias

Creates a numbered version from DRAFT, then creates an alias pointing to it with `en_US` enabled.

In [ ]:
# Step 1: Create a numbered bot version from DRAFT
print('⏳ Creating bot version...')
version_response = lex_client.create_bot_version(
    botId=BOT_ID,
    botVersionLocaleSpecification={
        LOCALE_ID: {
            'sourceBotVersion': 'DRAFT'
        }
    }
)
BOT_VERSION_NUM = version_response['botVersion']
print(f'   Bot version created: {BOT_VERSION_NUM}')

# Wait before polling to allow version to register
print('⏳ Waiting 15 seconds before polling...')
time.sleep(15)

# Poll until version is available
while True:
    try:
        v_status = lex_client.describe_bot_version(
            botId=BOT_ID,
            botVersion=BOT_VERSION_NUM
        )['botStatus']
        print(f'   Version status: {v_status}')
        if v_status == 'Available':
            print('✅ Version available!')
            break
        elif v_status in ['Failed', 'Deleting']:
            print(f'❌ Version failed: {v_status}')
            break
    except botocore.exceptions.ClientError as e:
        if e.response['Error']['Code'] == 'ResourceNotFoundException':
            print('   Version not registered yet, retrying...')
        else:
            raise
    time.sleep(10)

# Step 2: Create alias with en_US locale enabled
response = lex_client.create_bot_alias(
    botAliasName='TestAlias',
    botId=BOT_ID,
    botVersion=BOT_VERSION_NUM,
    description='Alias for testing mySampleChatBot',
    botAliasLocaleSettings={
        'en_US': {
            'enabled': True
        }
    }
)

BOT_ALIAS_ID = response['botAliasId']
print(f'✅ Bot alias created! (Alias ID: {BOT_ALIAS_ID})')
print('⏳ Waiting 10 seconds for alias to become available...')
time.sleep(10)
print('✅ Ready to test!')

## Step 8 — Test the Bot

Sends test messages and prints the detected intent + bot response.

In [ ]:
runtime_client = boto3.client('lexv2-runtime', region_name='us-east-1')
SESSION_ID = 'test-session-001'

test_messages = [
    'Hello',
    'Hi',
    'I want to order 1 pizza',
    'I want to order 2 Burgers',
    'What is the weather today?',  # fallback
]

print('=' * 55)
print('🧪 CHATBOT TEST RESULTS')
print('=' * 55)

for msg in test_messages:
    try:
        response = runtime_client.recognize_text(
            botId=BOT_ID,
            botAliasId=BOT_ALIAS_ID,
            localeId=LOCALE_ID,
            sessionId=SESSION_ID,
            text=msg
        )
        intent = response['sessionState'].get('intent', {}).get('name', 'N/A')
        messages = response.get('messages', [])
        bot_reply = messages[0]['content'] if messages else '(no response)'

    except Exception as e:
        intent = 'ERROR'
        bot_reply = str(e)

    print(f'\n👤 User  : {msg}')
    print(f'🎯 Intent: {intent}')
    print(f'🤖 Bot   : {bot_reply}')

print('\n' + '=' * 55)
print('✅ Testing complete!')

## Step 9 — Cleanup

> ⚠️ Run this only when you are done. This permanently deletes the bot and alias.
> The SageMaker execution role is **not** deleted (we didn't create it).

In [ ]:
# Delete alias first
lex_client.delete_bot_alias(botAliasId=BOT_ALIAS_ID, botId=BOT_ID)
print(f'🗑️  Bot alias deleted: {BOT_ALIAS_ID}')

time.sleep(5)

# Delete the bot (also removes intents and locale)
lex_client.delete_bot(botId=BOT_ID)
print(f'🗑️  Bot deleted: {BOT_NAME} ({BOT_ID})')

print('\n✅ Cleanup complete. All Lex resources removed.')

## Summary

| Step | Action |
|---|---|
| 0 | Installed and imported `boto3` |
| 1 | Retrieved SageMaker execution role ARN |
| 2 | Created bot `mySampleChatBot` |
| 3 | Configured English locale (`en_US`) |
| 4 | Created `GreetingIntent` |
| 5 | Created `OrderFoodIntent` |
| 5b | Configured `FallbackIntent` with a response message |
| 6 | Built the bot |
| 7 | Created numbered bot version and alias (with `en_US` enabled) |
| 8 | Tested the bot with sample messages |
| 9 | Cleaned up all resources |

---
*Copyright © K21Academy | All Rights Reserved*